In [1]:
# ============================================================
# PHASE 4B.2 — AMAZON METADATA FEATURE EXTRACTION
# + ISOLATION FOREST DEPLOYMENT MODEL
# ============================================================

import os
import re
import json
import ast
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

print("=" * 80)
print("PHASE 4B.2 — AMAZON METADATA PRODUCTION PIPELINE")
print("=" * 80)

PHASE 4B.2 — AMAZON METADATA PRODUCTION PIPELINE


In [3]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR = PROJECT_ROOT / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = RAW_DIR / "amazon_electronics_metadata_sample.parquet"

FEATURE_REPORT_PATH = REPORT_DIR / "phase4b2_feature_quality.csv"
SUMMARY_PATH = REPORT_DIR / "phase4b2_final_summary.json"

FEATURE_DATA_PATH = RAW_DIR / "amazon_metadata_26_features.csv.gz"

MODEL_PATH = MODEL_DIR / "trustguard_metadata_isolation_forest.joblib"

print("Project root :", PROJECT_ROOT)
print("Metadata     :", METADATA_PATH)
print("Model output :", MODEL_PATH)

Project root : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml
Metadata     : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\amazon_electronics_metadata_sample.parquet
Model output : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_metadata_isolation_forest.joblib


In [4]:
# ============================================================
# LOAD AMAZON METADATA
# ============================================================

metadata = pd.read_parquet(METADATA_PATH)

print("=" * 80)
print("METADATA DATASET")
print("=" * 80)

print("Shape:", metadata.shape)

print("\nColumns:")
for i, col in enumerate(metadata.columns, 1):
    print(f"{i:2}. {col}")

print("\nDtypes:")
print(metadata.dtypes)

METADATA DATASET
Shape: (10000, 16)

Columns:
 1. main_category
 2. title
 3. average_rating
 4. rating_number
 5. features
 6. description
 7. price
 8. images
 9. videos
10. store
11. categories
12. details
13. parent_asin
14. bought_together
15. subtitle
16. author

Dtypes:
main_category          str
title                  str
average_rating     float64
rating_number        int64
features            object
description         object
price                  str
images              object
videos              object
store                  str
categories          object
details                str
parent_asin            str
bought_together        str
subtitle               str
author                 str
dtype: object


In [5]:
# ============================================================
# BASIC SANITY CHECK
# ============================================================

required_columns = [
    "title",
    "description",
    "features",
    "categories",
    "images",
    "videos",
    "store",
    "price",
    "average_rating",
    "rating_number",
    "parent_asin"
]

missing_required = [
    c for c in required_columns
    if c not in metadata.columns
]

print("=" * 80)
print("REQUIRED COLUMN CHECK")
print("=" * 80)

if missing_required:
    print("MISSING:")
    for c in missing_required:
        print(" -", c)
else:
    print("All required metadata columns are present.")

REQUIRED COLUMN CHECK
All required metadata columns are present.


In [6]:
# ============================================================
# ROBUST NESTED OBJECT HELPERS
# ============================================================

def is_missing_value(value):
    """
    Safely determine whether a scalar value is missing.
    Handles None, NaN and pandas NA.
    """

    if value is None:
        return True

    try:
        result = pd.isna(value)

        if isinstance(result, (bool, np.bool_)):
            return bool(result)

    except Exception:
        pass

    return False


def clean_text(value):
    """
    Convert scalar/object to clean text.
    """

    if is_missing_value(value):
        return ""

    return str(value).strip()


def normalize_sequence(value):
    """
    Convert ndarray/list/tuple/scalar into a Python list.

    Important:
    We do NOT use len(value) blindly because dictionaries,
    strings and nested objects have different meanings.
    """

    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (list, tuple)):
        return list(value)

    return [value]


def valid_items(value):
    """
    Return non-empty, non-null items from an array/list.
    """

    items = normalize_sequence(value)

    output = []

    for item in items:

        if is_missing_value(item):
            continue

        if isinstance(item, str):
            if not item.strip():
                continue

        output.append(item)

    return output

In [7]:
# ============================================================
# FEATURES PARSING
# ============================================================

def parse_features(value):
    """
    Amazon 'features' is generally a numpy.ndarray of strings.

    Returns:
        list[str]
    """

    items = valid_items(value)

    result = []

    for item in items:

        if isinstance(item, dict):
            item = json.dumps(item, ensure_ascii=False)

        text = str(item).strip()

        if text:
            result.append(text)

    return result


def feature_count(value):
    return len(parse_features(value))


def feature_text_length(value):
    items = parse_features(value)

    if not items:
        return 0

    combined = " ".join(items)

    return len(combined)

In [8]:
# ============================================================
# CATEGORY PARSING
# ============================================================

def parse_categories(value):
    """
    Amazon 'categories' is generally a numpy.ndarray.

    Returns:
        list[str]
    """

    items = valid_items(value)

    result = []

    for item in items:

        text = str(item).strip()

        if text:
            result.append(text)

    return result


def category_count(value):
    return len(parse_categories(value))


def get_leaf_category(value):
    """
    Use the deepest/last available Amazon category as the
    category reference for price statistics.
    """

    categories = parse_categories(value)

    if not categories:
        return None

    return categories[-1]

In [9]:
# ============================================================
# IMAGE PARSING
# ============================================================

def parse_images(value):
    """
    Amazon 'images' is a dictionary such as:

    {
        'hi_res': ndarray,
        'large': ndarray,
        'thumb': ndarray,
        'variant': ndarray
    }

    We count actual image URLs, NOT dictionary keys.
    """

    if not isinstance(value, dict):
        return []

    # Prefer large images.
    urls = value.get("large", [])

    if not valid_items(urls):

        # Fallback to hi_res.
        urls = value.get("hi_res", [])

    if not valid_items(urls):

        # Final fallback to thumbnails.
        urls = value.get("thumb", [])

    result = []

    for item in valid_items(urls):

        text = str(item).strip()

        if text.startswith("http"):
            result.append(text)

    return result


def image_count(value):
    return len(parse_images(value))

In [10]:
# ============================================================
# VIDEO PARSING
# ============================================================

def parse_videos(value):
    """
    Amazon 'videos' is generally:

    {
        'title': ndarray,
        'url': ndarray,
        'user_id': ndarray
    }

    We count actual video URLs where available.
    """

    if not isinstance(value, dict):
        return []

    urls = value.get("url", [])

    result = []

    for item in valid_items(urls):

        text = str(item).strip()

        if text.startswith("http"):
            result.append(text)

    return result


def video_count(value):
    return len(parse_videos(value))


def has_videos(value):
    return int(video_count(value) > 0)

In [11]:
# ============================================================
# DESCRIPTION PARSING
# ============================================================

def parse_description(value):
    """
    Handles:
      - string
      - ndarray
      - list
      - tuple
      - None
    """

    if is_missing_value(value):
        return ""

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    if isinstance(value, np.ndarray):
        parts = valid_items(value)
        return " ".join(str(x) for x in parts)

    if isinstance(value, (list, tuple)):
        parts = valid_items(value)
        return " ".join(str(x) for x in parts)

    return str(value).strip()


def description_length(value):
    return len(parse_description(value))


def description_word_count(value):
    text = parse_description(value)

    if not text:
        return 0

    return len(re.findall(r"\b\w+\b", text))

In [13]:
# ============================================================
# TITLE FEATURES
# ============================================================

def title_text(value):
    return clean_text(value)


def title_length(value):
    return len(title_text(value))


def title_word_count(value):
    text = title_text(value)

    if not text:
        return 0

    return len(re.findall(r"\b\w+\b", text))


def uppercase_ratio(value):
    text = title_text(value)

    letters = [c for c in text if c.isalpha()]

    if not letters:
        return 0.0

    uppercase = sum(c.isupper() for c in letters)

    return uppercase / len(letters)


def special_character_ratio(value):
    text = title_text(value)

    if not text:
        return 0.0

    special = sum(
        1 for c in text
        if not c.isalnum() and not c.isspace()
    )

    return special / len(text)
    # ============================================================
# SELLER FEATURES
# ============================================================

def seller_missing(value):
    text = clean_text(value)

    return int(text == "")


def seller_name_length(value):
    text = clean_text(value)

    return len(text)

    # ============================================================
# PRICE PARSING
# ============================================================

def parse_price(value):
    """
    Convert Amazon price strings such as:

        '$19.99'
        '19.99'
        '$1,299.00'

    into numeric values.

    Invalid/missing values -> NaN
    """

    if is_missing_value(value):
        return np.nan

    # Handle numeric values directly.
    if isinstance(value, (int, float, np.integer, np.floating)):

        if np.isfinite(value):
            return float(value)

        return np.nan

    text = str(value).strip()

    if not text:
        return np.nan

    # Remove currency symbols and commas.
    cleaned = re.sub(r"[^\d.\-]", "", text)

    if not cleaned:
        return np.nan

    try:
        number = float(cleaned)

        if number < 0:
            return np.nan

        return number

    except Exception:
        return np.nan

In [14]:
# ============================================================
# DERIVE NON-PRICE FEATURES
# ============================================================

features_df = pd.DataFrame(index=metadata.index)

features_df["title_length"] = metadata["title"].apply(title_length)

features_df["title_word_count"] = metadata["title"].apply(
    title_word_count
)

features_df["uppercase_ratio"] = metadata["title"].apply(
    uppercase_ratio
)

features_df["special_character_ratio"] = metadata["title"].apply(
    special_character_ratio
)

features_df["description_length"] = metadata["description"].apply(
    description_length
)

features_df["description_word_count"] = metadata["description"].apply(
    description_word_count
)

features_df["feature_count"] = metadata["features"].apply(
    feature_count
)

features_df["feature_text_length"] = metadata["features"].apply(
    feature_text_length
)

features_df["category_count"] = metadata["categories"].apply(
    category_count
)

features_df["image_count"] = metadata["images"].apply(
    image_count
)

features_df["video_count"] = metadata["videos"].apply(
    video_count
)

features_df["has_videos"] = metadata["videos"].apply(
    has_videos
)

features_df["seller_missing"] = metadata["store"].apply(
    seller_missing
)

features_df["seller_name_length"] = metadata["store"].apply(
    seller_name_length
)

features_df["price_numeric"] = metadata["price"].apply(
    parse_price
)

features_df["price_missing"] = features_df["price_numeric"].isna().astype(int)

features_df["average_rating"] = pd.to_numeric(
    metadata["average_rating"],
    errors="coerce"
)

features_df["rating_number"] = pd.to_numeric(
    metadata["rating_number"],
    errors="coerce"
)

print("Base feature matrix created.")
print("Shape:", features_df.shape)

Base feature matrix created.
Shape: (10000, 18)


In [15]:
# ============================================================
# NESTED PARSER VALIDATION
# ============================================================

print("=" * 80)
print("CORRECTED NESTED FEATURE DISTRIBUTIONS")
print("=" * 80)

nested_check = pd.DataFrame({
    "feature_count": features_df["feature_count"],
    "category_count": features_df["category_count"],
    "image_count": features_df["image_count"],
    "video_count": features_df["video_count"],
    "has_videos": features_df["has_videos"]
})

display(nested_check.describe())

print("\nUnique values:")

for col in nested_check.columns:
    print(
        f"{col:25s}: "
        f"{nested_check[col].nunique(dropna=True)} unique"
    )

print("\nImage count distribution:")
display(features_df["image_count"].value_counts().sort_index().head(20))

print("\nVideo count distribution:")
display(features_df["video_count"].value_counts().sort_index().head(20))

CORRECTED NESTED FEATURE DISTRIBUTIONS


,feature_count,category_count,image_count,video_count,has_videos
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000
mean,3.735900,4.107100,5.27590,2.655200,0.422200
std,2.290082,1.462547,2.71122,3.899645,0.493935
min,0.000000,0.000000,1.00000,0.000000,0.000000
25%,2.000000,4.000000,3.00000,0.000000,0.000000
50%,5.000000,4.000000,6.00000,0.000000,0.000000
75%,5.000000,5.000000,7.00000,5.000000,1.000000
max,19.000000,7.000000,26.00000,10.000000,1.000000



Unique values:
feature_count            : 18 unique
category_count           : 7 unique
image_count              : 22 unique
video_count              : 11 unique
has_videos               : 2 unique

Image count distribution:


image_count
1     1573
2      635
3      673
4      758
5      867
6     1313
7     2180
8      921
9      985
10      39
11      18
12       9
13       5
14       7
15       5
16       4
17       3
18       1
21       1
22       1
Name: count, dtype: int64


Video count distribution:


video_count
0     5778
1      674
2      381
3      290
4      258
5      207
6      222
7      185
8      176
9      146
10    1683
Name: count, dtype: int64

In [17]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# ============================================================
# PARSED EXAMPLES
# ============================================================

print("=" * 80)
print("PARSED NESTED EXAMPLES")
print("=" * 80)

for idx in range(min(10, len(metadata))):
    print(f"\nPRODUCT {idx}")
    print("-" * 60)
    print("Features:", parse_features(metadata.loc[idx, "features"])[:3])
    print("Feature count:", feature_count(metadata.loc[idx, "features"]))
    print("Categories:", parse_categories(metadata.loc[idx, "categories"]))
    print("Category count:", category_count(metadata.loc[idx, "categories"]))
    print("Image count:", image_count(metadata.loc[idx, "images"]))
    print("Video count:", video_count(metadata.loc[idx, "videos"]))

# ============================================================
# CATEGORY REFERENCE
# ============================================================

features_df["leaf_category"] = metadata["categories"].apply(get_leaf_category)

print("=" * 80)
print("CATEGORY REFERENCE")
print("=" * 80)

print("Products with category:", features_df["leaf_category"].notna().sum())
print("Missing category:", features_df["leaf_category"].isna().sum())
print("Unique leaf categories:", features_df["leaf_category"].nunique(dropna=True))

display(features_df["leaf_category"].value_counts(dropna=False).head(20))

# ============================================================
# REFERENCE / VALIDATION SPLIT INDICES
# ============================================================

RANDOM_STATE = 42

indices = np.arange(len(features_df))
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(indices)

split_point = int(len(indices) * 0.80)

train_idx = indices[:split_point]
validation_idx = indices[split_point:]

# Temporary split to calculate training-only category medians
train_temp = features_df.iloc[train_idx]

# ============================================================
# CATEGORY PRICE STATISTICS (TRAINING DATA ONLY)
# ============================================================

global_price_median = train_temp["price_numeric"].median()

category_price_median = (
    train_temp.dropna(subset=["price_numeric"])
    .groupby("leaf_category")["price_numeric"]
    .median()
)

print("=" * 80)
print("CATEGORY PRICE STATISTICS")
print("=" * 80)

print("Global training median price:", global_price_median)
print("Categories with learned price median:", len(category_price_median))

display(category_price_median.head(20))

# ============================================================
# PRICE-DERIVED FEATURES
# ============================================================

def get_category_price_reference(row):
    """
    Use category-specific median learned ONLY from training data.
    Fallback: global training median
    """
    category = row["leaf_category"]

    if pd.notna(category):
        value = category_price_median.get(category, np.nan)
        if pd.notna(value) and value > 0:
            return value

    return global_price_median

features_df["category_price_reference"] = features_df.apply(
    get_category_price_reference, axis=1
)

features_df["price_ratio_to_category"] = (
    features_df["price_numeric"] / features_df["category_price_reference"]
)

features_df["log_price_ratio"] = np.log1p(features_df["price_ratio_to_category"])

# Price anomaly: absolute log-distance from category median.
features_df["price_anomaly"] = np.abs(features_df["log_price_ratio"])

# ============================================================
# RATING FEATURES
# ============================================================

features_df["log_rating_number"] = np.log1p(
    features_df["rating_number"].clip(lower=0)
)

# Distance from the neutral midpoint of the Amazon 1-5 rating scale.
features_df["rating_extremeness"] = np.abs(features_df["average_rating"] - 3.0)

features_df["high_rating"] = (
    features_df["average_rating"] >= 4.5
).astype(int)

features_df["low_review_count"] = (
    features_df["rating_number"] < 10
).astype(int)

features_df["high_rating_low_reviews"] = (
    (features_df["high_rating"] == 1) & (features_df["low_review_count"] == 1)
).astype(int)

# ============================================================
# CREATE TRAIN / VALIDATION DATAFRAMES (POST FEATURE-CREATION)
# ============================================================

train_df = features_df.iloc[train_idx].copy()
validation_df = features_df.iloc[validation_idx].copy()

print("=" * 80)
print("REFERENCE SPLIT")
print("=" * 80)
print("Training/reference products :", len(train_df))
print("Validation products         :", len(validation_df))

# ============================================================
# FINAL TRUSTGUARD 26-FEATURE SPACE
# ============================================================

FEATURE_COLUMNS = [
    "title_length",
    "title_word_count",
    "uppercase_ratio",
    "special_character_ratio",
    "description_length",
    "description_word_count",
    "feature_count",
    "feature_text_length",
    "category_count",
    "image_count",
    "video_count",
    "has_videos",
    "seller_missing",
    "seller_name_length",
    "price_numeric",
    "price_missing",
    "price_ratio_to_category",
    "log_price_ratio",
    "price_anomaly",
    "average_rating",
    "rating_number",
    "log_rating_number",
    "rating_extremeness",
    "high_rating",
    "low_review_count",
    "high_rating_low_reviews",
]

print("=" * 80)
print("FEATURE SPACE")
print("=" * 80)

print("Total features:", len(FEATURE_COLUMNS))

for i, feature in enumerate(FEATURE_COLUMNS, 1):
    print(f"{i:2}. {feature}")

# ============================================================
# FINAL FEATURE MATRIX
# ============================================================

X_raw = features_df[FEATURE_COLUMNS].copy()

print("=" * 80)
print("FINAL FEATURE MATRIX")
print("=" * 80)

print("Shape:", X_raw.shape)
display(X_raw.head())

# ============================================================
# FEATURE QUALITY AUDIT
# ============================================================

quality_rows = []

for feature in FEATURE_COLUMNS:
    series = X_raw[feature]
    missing_pct = series.isna().mean() * 100
    unique_count = series.nunique(dropna=True)

    if unique_count <= 1:
        status = "CONSTANT"
    elif missing_pct > 50:
        status = "HIGH_MISSINGNESS"
    elif missing_pct > 10:
        status = "MODERATE_MISSINGNESS"
    else:
        status = "USABLE"

    quality_rows.append(
        {
            "feature": feature,
            "missing_pct": round(missing_pct, 4),
            "unique_count": int(unique_count),
            "mean": series.mean(),
            "std": series.std(),
            "min": series.min(),
            "max": series.max(),
            "status": status,
        }
    )

quality_df = pd.DataFrame(quality_rows)

print("=" * 80)
print("FEATURE QUALITY REPORT")
print("=" * 80)

display(quality_df)

quality_df.to_csv(FEATURE_REPORT_PATH, index=False)
print("\nSaved:", FEATURE_REPORT_PATH)

# ============================================================
# MODEL FEATURE SELECTION
# ============================================================

model_features = quality_df[
    quality_df["status"].isin(["USABLE", "MODERATE_MISSINGNESS"])
]["feature"].tolist()

constant_features = quality_df[quality_df["status"] == "CONSTANT"][
    "feature"
].tolist()

high_missing_features = quality_df[
    quality_df["status"] == "HIGH_MISSINGNESS"
]["feature"].tolist()

print("=" * 80)
print("MODEL FEATURE SELECTION")
print("=" * 80)

print("Official features :", len(FEATURE_COLUMNS))
print("Model features    :", len(model_features))

print("\nConstant:")
print(constant_features)

print("\nHigh missingness:")
print(high_missing_features)

print("\nModel features:")
for f in model_features:
    print(" -", f)

# ============================================================
# TRAINING-ONLY IMPUTATION
# ============================================================

imputer = SimpleImputer(strategy="median")

X_train_raw = train_df[model_features].copy()
X_train = imputer.fit_transform(X_train_raw)

print("=" * 80)
print("IMPUTATION")
print("=" * 80)

print("Original shape :", X_train_raw.shape)
print("Imputed shape  :", X_train.shape)
print("Remaining NaN  :", np.isnan(X_train).sum())

PARSED NESTED EXAMPLES

PRODUCT 0
------------------------------------------------------------
Features: []
Feature count: 0
Categories: ['Electronics', 'Television & Video', 'Video Glasses']
Category count: 3
Image count: 1
Video count: 0

PRODUCT 1
------------------------------------------------------------
Features: ['UPC: 662774021904', 'Weight: 0.600 lbs']
Feature count: 2
Categories: ['Electronics', 'Television & Video', 'Accessories', 'Cables', 'HDMI Cables']
Category count: 5
Image count: 5
Video count: 0

PRODUCT 2
------------------------------------------------------------
Features: ['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).', 'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, water and fingerprint. Your Macbook remains fresh some years later.', 'Elegant Style. Our stylish design and pri

leaf_category
Cases                          903
NaN                            724
Earbud Headphones              242
USB Cables                     184
Backgrounds                    170
Remote Controls                167
Traditional Laptops            166
Sleeves                        143
Batteries                      138
Backpacks                      135
Arm & Wristband Accessories    131
Skins & Decals                 111
Chargers & Adapters            110
AC Adapters                    103
Screen Protectors               97
Camera Cases                    96
Stands                          90
Memory                          88
USB Flash Drives                85
Portable Bluetooth Speakers     82
Name: count, dtype: int64

CATEGORY PRICE STATISTICS
Global training median price: 20.52
Categories with learned price median: 445


leaf_category
2 in 1 Laptops                  899.990
3D Glasses                        8.990
6 Month Financing               499.990
AC Adapters                      12.990
Access-Control Keypads           56.480
Accessories                      14.990
Accessories & Supplies           37.950
Accessory Bundles                21.490
Accessory Kits                   36.920
Adapter Rings                    40.000
Adapters                         12.990
Adapters & Converters           153.400
Aircraft Accessories             75.285
All-in-Ones                     426.310
Amplifiers                       92.990
Antennas                         21.990
Anti-Glare & Privacy Filters     49.990
Arm & Wristband Accessories      10.990
Armbands                         24.990
Audio & Video Accessories       137.495
Name: price_numeric, dtype: float64

REFERENCE SPLIT
Training/reference products : 8000
Validation products         : 2000
FEATURE SPACE
Total features: 26
 1. title_length
 2. title_word_count
 3. uppercase_ratio
 4. special_character_ratio
 5. description_length
 6. description_word_count
 7. feature_count
 8. feature_text_length
 9. category_count
10. image_count
11. video_count
12. has_videos
13. seller_missing
14. seller_name_length
15. price_numeric
16. price_missing
17. price_ratio_to_category
18. log_price_ratio
19. price_anomaly
20. average_rating
21. rating_number
22. log_rating_number
23. rating_extremeness
24. high_rating
25. low_review_count
26. high_rating_low_reviews
FINAL FEATURE MATRIX
Shape: (10000, 26)


,title_length,title_word_count,uppercase_ratio,special_character_ratio,description_length,description_word_count,feature_count,feature_text_length,category_count,image_count,...,price_ratio_to_category,log_price_ratio,price_anomaly,average_rating,rating_number,log_rating_number,rating_extremeness,high_rating,low_review_count,high_rating_low_reviews
0,38,6,1.000000,0.026316,658,105,0,0,3,1,...,NaN,NaN,NaN,3.5,6,1.945910,0.5,0,1,0
1,29,6,0.500000,0.068966,18,4,2,35,5,5,...,NaN,NaN,NaN,5.0,1,0.693147,2.0,1,1,1
2,202,31,0.194631,0.049505,0,0,5,823,5,6,...,2.327125,1.202108,1.202108,4.5,246,5.509388,1.5,1,0,0
3,199,28,0.142857,0.015075,0,0,5,1102,3,6,...,0.644932,0.497699,0.497699,4.5,233,5.455321,1.5,1,0,0
4,38,6,0.181818,0.000000,242,34,3,142,5,5,...,0.901925,0.642867,0.642867,3.8,64,4.174387,0.8,0,0,0


FEATURE QUALITY REPORT


,feature,missing_pct,unique_count,mean,std,min,max,status
0,title_length,0.00,266,127.208400,55.523979,3.000000,534.000000,USABLE
1,title_word_count,0.00,69,21.444500,9.785290,1.000000,95.000000,USABLE
2,uppercase_ratio,0.00,2404,0.264765,0.122268,0.000000,1.000000,USABLE
3,special_character_ratio,0.00,1338,0.040271,0.027267,0.000000,0.225806,USABLE
4,description_length,0.00,1957,480.851900,920.807882,0.000000,21055.000000,USABLE
5,description_word_count,0.00,551,80.671900,153.779812,0.000000,3417.000000,USABLE
6,feature_count,0.00,18,3.735900,2.290082,0.000000,19.000000,USABLE
7,feature_text_length,0.00,1898,544.194000,544.620353,0.000000,4216.000000,USABLE
8,category_count,0.00,7,4.107100,1.462547,0.000000,7.000000,USABLE
9,image_count,0.00,22,5.275900,2.711220,1.000000,26.000000,USABLE



Saved: C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b2_feature_quality.csv
MODEL FEATURE SELECTION
Official features : 26
Model features    : 22

Constant:
[]

High missingness:
['price_numeric', 'price_ratio_to_category', 'log_price_ratio', 'price_anomaly']

Model features:
 - title_length
 - title_word_count
 - uppercase_ratio
 - special_character_ratio
 - description_length
 - description_word_count
 - feature_count
 - feature_text_length
 - category_count
 - image_count
 - video_count
 - has_videos
 - seller_missing
 - seller_name_length
 - price_missing
 - average_rating
 - rating_number
 - log_rating_number
 - rating_extremeness
 - high_rating
 - low_review_count
 - high_rating_low_reviews
IMPUTATION
Original shape : (8000, 22)
Imputed shape  : (8000, 22)
Remaining NaN  : 0


In [18]:
# ============================================================
# ISOLATION FOREST
# ============================================================

ISOLATION_CONTAMINATION = 0.05

isolation_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination=ISOLATION_CONTAMINATION,
    max_features=1.0,
    bootstrap=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

isolation_forest.fit(X_train)

print("=" * 80)
print("ISOLATION FOREST TRAINING")
print("=" * 80)

print("Model:", isolation_forest)
print("\nTraining complete.")

ISOLATION FOREST TRAINING
Model: IsolationForest(contamination=0.05, n_estimators=300, n_jobs=-1,
                random_state=42)

Training complete.


In [20]:
# ============================================================
# ISOLATION FOREST SCORES
# ============================================================

train_raw = train_df[model_features].copy()

train_imputed = imputer.transform(train_raw)

# sklearn:
# decision_function -> higher = more normal
# We invert it so higher = more anomalous.

train_normality = isolation_forest.decision_function(
    train_imputed
)

train_anomaly = -train_normality

print("=" * 80)
print("ANOMALY SCORE DISTRIBUTION")
print("=" * 80)

print(pd.Series(train_anomaly).describe())

ANOMALY SCORE DISTRIBUTION
count    8000.000000
mean       -0.068552
std         0.037254
min        -0.145634
25%        -0.096572
50%        -0.072714
75%        -0.045394
max         0.118544
dtype: float64


In [21]:
# ============================================================
# INTERPRETABLE RISK INDICATORS
# ============================================================

def compute_risk_indicators(df):
    """
    Rule-based indicators.

    These are NOT ground-truth fraud labels.
    They are interpretable risk signals.
    """

    out = pd.DataFrame(index=df.index)

    # --------------------------------------------------------
    # 1. Seller risk
    # --------------------------------------------------------

    out["risk_missing_seller"] = (
        df["seller_missing"] == 1
    ).astype(int)

    # --------------------------------------------------------
    # 2. Sparse content
    # --------------------------------------------------------

    out["risk_no_description"] = (
        df["description_length"] == 0
    ).astype(int)

    out["risk_no_features"] = (
        df["feature_count"] == 0
    ).astype(int)

    # --------------------------------------------------------
    # 3. Sparse media
    # --------------------------------------------------------

    out["risk_no_images"] = (
        df["image_count"] == 0
    ).astype(int)

    out["risk_no_videos"] = (
        df["has_videos"] == 0
    ).astype(int)

    # --------------------------------------------------------
    # 4. Rating manipulation-style signals
    # --------------------------------------------------------

    out["risk_high_rating_low_reviews"] = (
        df["high_rating_low_reviews"] == 1
    ).astype(int)

    # --------------------------------------------------------
    # 5. Extreme rating
    # --------------------------------------------------------

    out["risk_extreme_rating"] = (
        df["rating_extremeness"] >= 1.5
    ).astype(int)

    # --------------------------------------------------------
    # 6. Aggressive title formatting
    # --------------------------------------------------------

    out["risk_high_uppercase"] = (
        df["uppercase_ratio"] >= 0.50
    ).astype(int)

    out["risk_high_special_chars"] = (
        df["special_character_ratio"] >= 0.25
    ).astype(int)

    # --------------------------------------------------------
    # 7. Price anomaly
    # --------------------------------------------------------

    out["risk_price_anomaly"] = (
        df["price_anomaly"] >= 1.0
    ).astype(int)

    # --------------------------------------------------------
    # 8. Low review volume
    # --------------------------------------------------------

    out["risk_low_review_count"] = (
        df["low_review_count"] == 1
    ).astype(int)

    return out

In [22]:
# ============================================================
# RISK INDICATORS
# ============================================================

risk_indicators = compute_risk_indicators(
    train_df[FEATURE_COLUMNS]
)

print("=" * 80)
print("RISK INDICATOR DISTRIBUTION")
print("=" * 80)

risk_summary = pd.DataFrame({
    "indicator": risk_indicators.columns,
    "count": risk_indicators.sum().values,
    "percentage": (
        risk_indicators.mean().values * 100
    )
})

risk_summary["percentage"] = risk_summary[
    "percentage"
].round(2)

display(risk_summary)

RISK INDICATOR DISTRIBUTION


,indicator,count,percentage
0,risk_missing_seller,48,0.60
1,risk_no_description,3381,42.26
2,risk_no_features,1762,22.02
3,risk_no_images,0,0.00
4,risk_no_videos,4622,57.78
5,risk_high_rating_low_reviews,1179,14.74
6,risk_extreme_rating,2941,36.76
7,risk_high_uppercase,406,5.08
8,risk_high_special_chars,0,0.00
9,risk_price_anomaly,682,8.52


In [23]:
# ============================================================
# RULE-BASED RISK SCORE
# ============================================================

RISK_WEIGHTS = {
    "risk_missing_seller": 12,
    "risk_no_description": 8,
    "risk_no_features": 6,
    "risk_no_images": 5,
    "risk_no_videos": 2,
    "risk_high_rating_low_reviews": 20,
    "risk_extreme_rating": 5,
    "risk_high_uppercase": 4,
    "risk_high_special_chars": 4,
    "risk_price_anomaly": 15,
    "risk_low_review_count": 8
}

rule_score = np.zeros(len(risk_indicators))

for indicator, weight in RISK_WEIGHTS.items():

    rule_score += (
        risk_indicators[indicator].values *
        weight
    )

# Cap at 100.
rule_score = np.clip(rule_score, 0, 100)

print("=" * 80)
print("RULE-BASED RISK SCORE")
print("=" * 80)

print(
    pd.Series(rule_score).describe()
)

RULE-BASED RISK SCORE
count    8000.000000
mean       15.014375
std        13.230563
min         0.000000
25%         5.000000
50%        10.000000
75%        21.000000
max        68.000000
dtype: float64


In [24]:
# ============================================================
# ANOMALY SCORE -> 0-100
# ============================================================

def empirical_percentile(value, reference_values):
    """
    Percentage of reference observations <= value.
    """

    reference_values = np.asarray(reference_values)

    return (
        np.searchsorted(
            np.sort(reference_values),
            value,
            side="right"
        )
        / len(reference_values)
    )


train_anomaly_score_100 = np.array([
    empirical_percentile(x, train_anomaly)
    for x in train_anomaly
]) * 100

print("=" * 80)
print("ISOLATION FOREST RISK SCORE")
print("=" * 80)

print(
    pd.Series(train_anomaly_score_100).describe()
)

ISOLATION FOREST RISK SCORE
count    8000.000000
mean       50.006253
std        28.869316
min         0.012500
25%        25.009375
50%        50.006250
75%        75.003125
max       100.000000
dtype: float64


In [25]:
# ============================================================
# COMBINED PRODUCTION RISK SCORE
# ============================================================

ANOMALY_WEIGHT = 0.60
RULE_WEIGHT = 0.40

combined_risk_score = (
    ANOMALY_WEIGHT * train_anomaly_score_100 +
    RULE_WEIGHT * rule_score
)

combined_risk_score = np.clip(
    combined_risk_score,
    0,
    100
)

print("=" * 80)
print("COMBINED RISK SCORE")
print("=" * 80)

print(
    pd.Series(combined_risk_score).describe()
)

COMBINED RISK SCORE
count    8000.000000
mean       36.009502
std        20.045364
min         0.845000
25%        19.081250
50%        34.630000
75%        52.713125
max        85.745000
dtype: float64


In [26]:
# ============================================================
# RISK BANDS
# ============================================================

def risk_band(score):

    if score < 25:
        return "LOW"

    elif score < 50:
        return "MEDIUM"

    elif score < 75:
        return "HIGH"

    else:
        return "CRITICAL"


risk_bands = pd.Series(
    combined_risk_score
).apply(risk_band)

print("=" * 80)
print("RISK BAND DISTRIBUTION")
print("=" * 80)

print(
    risk_bands.value_counts()
)

RISK BAND DISTRIBUTION
MEDIUM      2916
LOW         2799
HIGH        2183
CRITICAL     102
Name: count, dtype: int64


In [29]:
# ============================================================
# TOP ANOMALOUS PRODUCTS
# ============================================================

# Copy only the training rows from metadata to match the 8,000 index & keep metadata columns
inspection_df = metadata.iloc[train_idx].copy()

inspection_df["anomaly_score"] = train_anomaly_score_100
inspection_df["rule_risk_score"] = rule_score
inspection_df["risk_score"] = combined_risk_score
inspection_df["risk_band"] = risk_bands.values

inspection_df["risk_band"] = pd.Categorical(
    inspection_df["risk_band"],
    categories=["LOW", "MEDIUM", "HIGH", "CRITICAL"],
    ordered=True,
)

top_risky = inspection_df.sort_values("risk_score", ascending=False).head(20)

display(
    top_risky[
        [
            "parent_asin",
            "title",
            "store",
            "price",
            "average_rating",
            "rating_number",
            "anomaly_score",
            "rule_risk_score",
            "risk_score",
            "risk_band",
        ]
    ]
)

,parent_asin,title,store,price,average_rating,rating_number,anomaly_score,rule_risk_score,risk_score,risk_band
1665,B00AMKMB2I,Las Vegas HD Video Tour (BLU-RAY DISC),NaN,None,5.0,1,99.5750,65.0,85.7450,CRITICAL
4002,B00HJEJU5Q,"EBM PAPST W2E142-BB05-01 AXIAL FAN, 150MM x 17...",EBM Papst,170.72,4.6,3,97.4000,68.0,85.6400,CRITICAL
9542,B072MNP6D8,"Fire 7 Kids Edition (7th Gen, 2017 Release) Sc...",NaN,9.59,5.0,3,99.3625,61.0,84.0175,CRITICAL
5838,B01INITER0,Cambond 2 Pack Apple Watch Band 42mm Series 2 ...,NaN,None,4.5,2,99.0625,61.0,83.8375,CRITICAL
9823,B07NXXG838,Cisco SG550X-24 24-Port Gigabit Stackable Mana...,Cisco,719.99,5.0,1,98.1000,62.0,83.6600,CRITICAL
5677,B0977QTY56,"HP Elitebook 840 G5 14"" Full HD FHD Business L...",HP,259.99,5.0,1,96.6125,64.0,83.5675,CRITICAL
9567,B07JWVC4YG,Echo Dot (3rd Gen) - Charcoal with Tile Mate -...,NaN,None,5.0,1,98.5250,61.0,83.5150,CRITICAL
2532,B01BD2I7GC,"Caison 13.3"" Laptop Sleeve Cross Body Case Bag...",NaN,None,5.0,1,97.5000,61.0,82.9000,CRITICAL
5220,B012LASAYE,Pichon 15 Inch Multi Purpose Laptop and Tablet...,NaN,None,4.5,5,97.4875,61.0,82.8925,CRITICAL
4171,B01MCYYQEB,NEWSTYLE Case for All-New Amazon Fire HD 8 (20...,NaN,None,5.0,4,97.3750,61.0,82.8250,CRITICAL


In [30]:
# ============================================================
# FEATURE REDUNDANCY AUDIT
# ============================================================

numeric_quality = X_raw.copy()

correlation = numeric_quality.corr(
    numeric_only=True
)

high_corr_pairs = []

for i in range(len(correlation.columns)):

    for j in range(i + 1, len(correlation.columns)):

        a = correlation.columns[i]
        b = correlation.columns[j]

        value = correlation.iloc[i, j]

        if pd.notna(value) and abs(value) >= 0.90:

            high_corr_pairs.append({
                "feature_1": a,
                "feature_2": b,
                "correlation": value
            })

high_corr_df = pd.DataFrame(high_corr_pairs)

print("=" * 80)
print("HIGH-CORRELATION FEATURE PAIRS")
print("=" * 80)

display(high_corr_df)

HIGH-CORRELATION FEATURE PAIRS


,feature_1,feature_2,correlation
0,title_length,title_word_count,0.954179
1,description_length,description_word_count,0.997904
2,log_price_ratio,price_anomaly,1.000000


In [31]:
# ============================================================
# SAVE ENGINEERED FEATURE DATASET
# ============================================================

production_features = X_raw.copy()

production_features.insert(
    0,
    "parent_asin",
    metadata["parent_asin"].values
)

production_features.to_csv(
    FEATURE_DATA_PATH,
    index=False,
    compression="gzip"
)

print("=" * 80)
print("FEATURE DATASET EXPORTED")
print("=" * 80)

print("Saved:", FEATURE_DATA_PATH)
print("Shape:", production_features.shape)

FEATURE DATASET EXPORTED
Saved: C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\amazon_metadata_26_features.csv.gz
Shape: (10000, 27)


In [32]:
# ============================================================
# FINAL DEPLOYMENT MODEL
# TRAIN ON ALL 10,000 PRODUCTS
# ============================================================

final_imputer = SimpleImputer(
    strategy="median"
)

X_all_raw = X_raw[model_features].copy()

X_all = final_imputer.fit_transform(
    X_all_raw
)

final_isolation_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination=ISOLATION_CONTAMINATION,
    max_features=1.0,
    bootstrap=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_isolation_forest.fit(X_all)

print("=" * 80)
print("FINAL DEPLOYMENT MODEL")
print("=" * 80)

print("Products used :", len(X_all))
print("Features used :", len(model_features))
print("Estimators    :", 300)
print("Contamination :", ISOLATION_CONTAMINATION)

print("\nTraining complete.")

FINAL DEPLOYMENT MODEL
Products used : 10000
Features used : 22
Estimators    : 300
Contamination : 0.05

Training complete.


In [33]:
# ============================================================
# FINAL DEPLOYMENT REFERENCES
# ============================================================

final_normality = final_isolation_forest.decision_function(
    X_all
)

final_anomaly_reference = -final_normality

# Final category-price statistics for deployment training set.
final_global_price_median = (
    X_raw["price_numeric"].median()
)

final_category_price_median = (
    features_df
    .dropna(subset=["price_numeric"])
    .groupby("leaf_category")["price_numeric"]
    .median()
)

print("=" * 80)
print("FINAL DEPLOYMENT REFERENCES")
print("=" * 80)

print(
    "Global price median:",
    final_global_price_median
)

print(
    "Category price references:",
    len(final_category_price_median)
)

print(
    "Anomaly reference observations:",
    len(final_anomaly_reference)
)

FINAL DEPLOYMENT REFERENCES
Global price median: 20.39
Category price references: 479
Anomaly reference observations: 10000


In [34]:
# ============================================================
# DEPLOYMENT INFERENCE PIPELINE
# ============================================================

def build_metadata_features(product):
    """
    Convert ONE raw Amazon metadata product dictionary
    into the exact TrustGuard 26-feature representation.

    This function must be used during production inference.
    """

    row = {}

    title = product.get("title")
    description = product.get("description")
    features = product.get("features")
    categories = product.get("categories")
    images = product.get("images")
    videos = product.get("videos")
    store = product.get("store")
    price = product.get("price")
    average_rating = product.get("average_rating")
    rating_number = product.get("rating_number")

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    row["title_length"] = title_length(title)

    row["title_word_count"] = title_word_count(title)

    row["uppercase_ratio"] = uppercase_ratio(title)

    row["special_character_ratio"] = (
        special_character_ratio(title)
    )

    row["description_length"] = (
        description_length(description)
    )

    row["description_word_count"] = (
        description_word_count(description)
    )

    # --------------------------------------------------------
    # Features / categories
    # --------------------------------------------------------

    row["feature_count"] = feature_count(features)

    row["feature_text_length"] = (
        feature_text_length(features)
    )

    row["category_count"] = category_count(categories)

    # --------------------------------------------------------
    # Media
    # --------------------------------------------------------

    row["image_count"] = image_count(images)

    row["video_count"] = video_count(videos)

    row["has_videos"] = has_videos(videos)

    # --------------------------------------------------------
    # Seller
    # --------------------------------------------------------

    row["seller_missing"] = seller_missing(store)

    row["seller_name_length"] = seller_name_length(store)

    # --------------------------------------------------------
    # Price
    # --------------------------------------------------------

    parsed_price = parse_price(price)

    row["price_numeric"] = parsed_price

    row["price_missing"] = int(pd.isna(parsed_price))

    # --------------------------------------------------------
    # Category price reference
    # --------------------------------------------------------

    leaf_category = get_leaf_category(categories)

    category_reference = (
        final_category_price_median.get(
            leaf_category,
            final_global_price_median
        )
        if leaf_category is not None
        else final_global_price_median
    )

    if (
        pd.isna(category_reference)
        or category_reference <= 0
        or pd.isna(parsed_price)
    ):

        row["price_ratio_to_category"] = np.nan
        row["log_price_ratio"] = np.nan
        row["price_anomaly"] = np.nan

    else:

        ratio = parsed_price / category_reference

        row["price_ratio_to_category"] = ratio

        row["log_price_ratio"] = np.log1p(ratio)

        row["price_anomaly"] = abs(
            row["log_price_ratio"]
        )

    # --------------------------------------------------------
    # Rating
    # --------------------------------------------------------

    row["average_rating"] = pd.to_numeric(
        average_rating,
        errors="coerce"
    )

    row["rating_number"] = pd.to_numeric(
        rating_number,
        errors="coerce"
    )

    rating_count = row["rating_number"]

    if pd.isna(rating_count):
        row["log_rating_number"] = np.nan
    else:
        row["log_rating_number"] = np.log1p(
            max(0, rating_count)
        )

    if pd.isna(row["average_rating"]):
        row["rating_extremeness"] = np.nan
        row["high_rating"] = 0
    else:
        row["rating_extremeness"] = abs(
            row["average_rating"] - 3.0
        )

        row["high_rating"] = int(
            row["average_rating"] >= 4.5
        )

    row["low_review_count"] = (
        int(
            pd.notna(rating_count)
            and rating_count < 10
        )
    )

    row["high_rating_low_reviews"] = int(
        row["high_rating"] == 1
        and row["low_review_count"] == 1
    )

    return pd.DataFrame(
        [row],
        columns=FEATURE_COLUMNS
    )

In [35]:
# ============================================================
# PRODUCTION RISK SCORER
# ============================================================

def anomaly_to_score(anomaly_value):
    """
    Convert raw anomaly value into 0-100 percentile
    using the final deployment reference population.
    """

    return (
        np.searchsorted(
            np.sort(final_anomaly_reference),
            anomaly_value,
            side="right"
        )
        / len(final_anomaly_reference)
    ) * 100


def score_product(product):
    """
    Complete production scoring pipeline.

    Returns:
        26 features
        anomaly score
        rule indicators
        combined risk score
        risk band
    """

    feature_row = build_metadata_features(product)

    # Model features.
    model_input_raw = feature_row[model_features]

    model_input = final_imputer.transform(
        model_input_raw
    )

    # Isolation Forest.
    normality = final_isolation_forest.decision_function(
        model_input
    )[0]

    anomaly_value = -normality

    anomaly_score = anomaly_to_score(
        anomaly_value
    )

    # Rule indicators.
    indicators = compute_risk_indicators(
        feature_row
    )

    rule_score = 0.0

    for indicator, weight in RISK_WEIGHTS.items():

        rule_score += (
            float(indicators.iloc[0][indicator])
            * weight
        )

    rule_score = min(
        max(rule_score, 0),
        100
    )

    # Combined.
    final_score = (
        ANOMALY_WEIGHT * anomaly_score +
        RULE_WEIGHT * rule_score
    )

    final_score = min(
        max(final_score, 0),
        100
    )

    result = {
        "anomaly_score": float(anomaly_score),
        "rule_risk_score": float(rule_score),
        "risk_score": float(final_score),
        "risk_band": risk_band(final_score)
    }

    for column in indicators.columns:
        result[column] = int(
            indicators.iloc[0][column]
        )

    return {
        "features": feature_row.iloc[0].to_dict(),
        "risk": result
    }

In [36]:
# ============================================================
# END-TO-END INFERENCE TEST
# ============================================================

test_product = metadata.iloc[0].to_dict()

result = score_product(
    test_product
)

print("=" * 80)
print("DEPLOYMENT INFERENCE TEST")
print("=" * 80)

print("\nProduct:")
print(test_product.get("title"))

print("\nRisk:")
print(
    json.dumps(
        result["risk"],
        indent=4
    )
)

DEPLOYMENT INFERENCE TEST

Product:
FS-1051 FATSHARK TELEPORTER V3 HEADSET

Risk:
{
    "anomaly_score": 85.05,
    "rule_risk_score": 20.0,
    "risk_score": 59.029999999999994,
    "risk_band": "HIGH",
    "risk_missing_seller": 0,
    "risk_no_description": 0,
    "risk_no_features": 1,
    "risk_no_images": 0,
    "risk_no_videos": 1,
    "risk_high_rating_low_reviews": 0,
    "risk_extreme_rating": 0,
    "risk_high_uppercase": 1,
    "risk_high_special_chars": 0,
    "risk_price_anomaly": 0,
    "risk_low_review_count": 1
}


In [37]:
# ============================================================
# FINAL FEATURE VECTOR VALIDATION
# ============================================================

output_features = result["features"]

print("=" * 80)
print("FEATURE VECTOR VALIDATION")
print("=" * 80)

print("Expected:", len(FEATURE_COLUMNS))
print("Actual  :", len(output_features))

missing_features = [
    f for f in FEATURE_COLUMNS
    if f not in output_features
]

extra_features = [
    f for f in output_features
    if f not in FEATURE_COLUMNS
]

print("\nMissing:", missing_features)
print("Extra  :", extra_features)

assert len(output_features) == 26
assert not missing_features
assert not extra_features

print("\n✓ Exact 26-feature contract verified.")

FEATURE VECTOR VALIDATION
Expected: 26
Actual  : 26

Missing: []
Extra  : []

✓ Exact 26-feature contract verified.


In [38]:
# ============================================================
# EXPORT COMPLETE DEPLOYMENT BUNDLE
# ============================================================

deployment_bundle = {

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    "model": final_isolation_forest,

    # --------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------

    "imputer": final_imputer,

    # --------------------------------------------------------
    # Feature contract
    # --------------------------------------------------------

    "feature_columns": FEATURE_COLUMNS,

    "model_features": model_features,

    "constant_features": constant_features,

    "high_missing_features": high_missing_features,

    # --------------------------------------------------------
    # Category price reference
    # --------------------------------------------------------

    "global_price_median": float(
        final_global_price_median
    ),

    "category_price_median": (
        final_category_price_median
        .to_dict()
    ),

    # --------------------------------------------------------
    # Risk configuration
    # --------------------------------------------------------

    "risk_weights": RISK_WEIGHTS,

    "anomaly_weight": ANOMALY_WEIGHT,

    "rule_weight": RULE_WEIGHT,

    "anomaly_reference": (
        final_anomaly_reference.tolist()
    ),

    # --------------------------------------------------------
    # Model metadata
    # --------------------------------------------------------

    "model_type": "IsolationForest",

    "n_estimators": 300,

    "contamination": ISOLATION_CONTAMINATION,

    "random_state": RANDOM_STATE,

    "dataset": (
        "amazon_electronics_metadata_sample.parquet"
    ),

    "training_rows": int(len(metadata)),

    "architecture": (
        "Amazon metadata -> "
        "26 engineered features -> "
        "Isolation Forest + "
        "interpretable risk indicators -> "
        "combined risk score"
    )
}

joblib.dump(
    deployment_bundle,
    MODEL_PATH
)

print("=" * 80)
print("MODEL EXPORT")
print("=" * 80)

print("Saved:")
print(MODEL_PATH)

print(
    "\nFile size:",
    round(
        MODEL_PATH.stat().st_size / (1024 * 1024),
        2
    ),
    "MB"
)

MODEL EXPORT
Saved:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_metadata_isolation_forest.joblib

File size: 3.85 MB


In [39]:
# ============================================================
# MODEL RELOAD TEST
# ============================================================

loaded_bundle = joblib.load(
    MODEL_PATH
)

print("=" * 80)
print("MODEL RELOAD TEST")
print("=" * 80)

print(
    "Loaded model:",
    type(loaded_bundle["model"])
)

print(
    "Feature count:",
    len(loaded_bundle["feature_columns"])
)

print(
    "Model feature count:",
    len(loaded_bundle["model_features"])
)

print("\n✓ Deployment bundle successfully reloaded.")

MODEL RELOAD TEST
Loaded model: <class 'sklearn.ensemble._iforest.IsolationForest'>
Feature count: 26
Model feature count: 22

✓ Deployment bundle successfully reloaded.


In [40]:
# ============================================================
# EXPORTED MODEL END-TO-END TEST
# ============================================================

loaded_model = loaded_bundle["model"]
loaded_imputer = loaded_bundle["imputer"]

loaded_feature_columns = (
    loaded_bundle["feature_columns"]
)

loaded_model_features = (
    loaded_bundle["model_features"]
)

# Rebuild one product.
test_features = build_metadata_features(
    test_product
)

test_input = loaded_imputer.transform(
    test_features[loaded_model_features]
)

loaded_normality = (
    loaded_model
    .decision_function(test_input)[0]
)

loaded_anomaly = -loaded_normality

print("=" * 80)
print("EXPORTED MODEL INFERENCE TEST")
print("=" * 80)

print("Product:", test_product.get("title"))

print("Raw anomaly:", loaded_anomaly)

print("\n✓ Exported artifact successfully performs inference.")

EXPORTED MODEL INFERENCE TEST
Product: FS-1051 FATSHARK TELEPORTER V3 HEADSET
Raw anomaly: -0.028701668630939592

✓ Exported artifact successfully performs inference.


In [41]:
# ============================================================
# FINAL PHASE 4B.2 SUMMARY
# ============================================================

usable_count = int(
    (quality_df["status"] == "USABLE").sum()
)

moderate_count = int(
    (quality_df["status"] == "MODERATE_MISSINGNESS").sum()
)

high_missing_count = int(
    (quality_df["status"] == "HIGH_MISSINGNESS").sum()
)

constant_count = int(
    (quality_df["status"] == "CONSTANT").sum()
)

final_summary = {

    "phase": "4B.2",

    "dataset": {
        "name": "amazon_electronics_metadata_sample.parquet",
        "rows": int(len(metadata)),
        "columns": int(len(metadata.columns))
    },

    "feature_space": {
        "total": 26,
        "usable": usable_count,
        "moderate_missingness": moderate_count,
        "high_missingness": high_missing_count,
        "constant": constant_count
    },

    "nested_parsing": {
        "features": "numpy.ndarray parsed",
        "categories": "numpy.ndarray parsed",
        "images": "nested dict -> actual image URLs counted",
        "videos": "nested dict -> actual video URLs counted"
    },

    "price_statistics": {
        "category_reference": "leaf category median",
        "learned_from_training_reference": True,
        "global_fallback": True
    },

    "model": {
        "type": "IsolationForest",
        "n_estimators": 300,
        "contamination": ISOLATION_CONTAMINATION,
        "random_state": RANDOM_STATE,
        "training_rows": int(len(metadata)),
        "model_features": len(model_features)
    },

    "risk_engine": {
        "anomaly_weight": ANOMALY_WEIGHT,
        "rule_weight": RULE_WEIGHT,
        "score_range": "0-100",
        "bands": [
            "LOW",
            "MEDIUM",
            "HIGH",
            "CRITICAL"
        ]
    },

    "ground_truth": {
        "used_for_production_training": False,
        "reason": (
            "Production engine is unsupervised. "
            "He-associated labels remain separate "
            "research benchmark."
        )
    },

    "artifacts": {
        "feature_dataset": str(FEATURE_DATA_PATH),
        "feature_quality_report": str(FEATURE_REPORT_PATH),
        "model_bundle": str(MODEL_PATH)
    }
}

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_summary,
        f,
        indent=4
    )

print("=" * 80)
print("PHASE 4B.2 — FINAL SUMMARY")
print("=" * 80)

print(
    json.dumps(
        final_summary,
        indent=4
    )
)

print("\nSaved:", SUMMARY_PATH)

PHASE 4B.2 — FINAL SUMMARY
{
    "phase": "4B.2",
    "dataset": {
        "name": "amazon_electronics_metadata_sample.parquet",
        "rows": 10000,
        "columns": 16
    },
    "feature_space": {
        "total": 26,
        "usable": 22,
        "moderate_missingness": 0,
        "high_missingness": 4,
        "constant": 0
    },
    "nested_parsing": {
        "features": "numpy.ndarray parsed",
        "categories": "numpy.ndarray parsed",
        "images": "nested dict -> actual image URLs counted",
        "videos": "nested dict -> actual video URLs counted"
    },
    "price_statistics": {
        "category_reference": "leaf category median",
        "learned_from_training_reference": true,
        "global_fallback": true
    },
    "model": {
        "type": "IsolationForest",
        "n_estimators": 300,
        "contamination": 0.05,
        "random_state": 42,
        "training_rows": 10000,
        "model_features": 22
    },
    "risk_engine": {
        "anomaly_we

In [42]:
# ============================================================
# FINAL PIPELINE CHECK
# ============================================================

print("=" * 80)
print("PHASE 4B.2 COMPLETE")
print("=" * 80)

print("\nDataset")
print("-------")
print("Amazon metadata rows :", len(metadata))

print("\nFeature Space")
print("-------------")
print("Total features      :", len(FEATURE_COLUMNS))
print("Model features      :", len(model_features))
print("Constant features   :", len(constant_features))
print("High-missing        :", len(high_missing_features))

print("\nModel")
print("-----")
print("Isolation Forest    : TRAINED")
print("Estimators          :", 300)
print("Contamination       :", ISOLATION_CONTAMINATION)

print("\nRisk Engine")
print("-----------")
print("Isolation Forest    : 60%")
print("Risk indicators     : 40%")
print("Final score         : 0-100")

print("\nArtifacts")
print("---------")
print("Feature dataset     :", FEATURE_DATA_PATH)
print("Quality report      :", FEATURE_REPORT_PATH)
print("Summary             :", SUMMARY_PATH)
print("Deployment model    :", MODEL_PATH)

print("\n✓ Phase 4B.2 pipeline completed successfully.")

PHASE 4B.2 COMPLETE

Dataset
-------
Amazon metadata rows : 10000

Feature Space
-------------
Total features      : 26
Model features      : 22
Constant features   : 0
High-missing        : 4

Model
-----
Isolation Forest    : TRAINED
Estimators          : 300
Contamination       : 0.05

Risk Engine
-----------
Isolation Forest    : 60%
Risk indicators     : 40%
Final score         : 0-100

Artifacts
---------
Feature dataset     : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\amazon_metadata_26_features.csv.gz
Quality report      : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b2_feature_quality.csv
Summary             : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b2_final_summary.json
Deployment model    : C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\models\trustguard_metadata_isolation_forest.joblib

✓ Phase 4B.2 pipeline completed successfully.
